In [ ]:
from pathlib import Path

def _sionna_repo_root():
    """Find project root when Jupyter is started from root or notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "scenes" / "University.xml").is_file():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory")


# Shared settings for exploratory cells
SCENE_NO_RIS = str(_sionna_repo_root() / "scenes" / "University.xml")
SCENE_WITH_RIS = str(_sionna_repo_root() / "scenes" / "University_RIS.xml")
FREQ_HZ = 1e9

from sionna.rt import load_scene

scene = load_scene(str(_sionna_repo_root() / "scenes" / "University.xml"))
print("scene loaded")
print("num objects =", len(scene.objects))
print(scene.objects.keys())

In [ ]:
scene.preview()

In [ ]:
scene = load_scene(SCENE_WITH_RIS)
print(scene.objects.keys())
print(scene.radio_materials.keys())

In [ ]:
from sionna.rt import ITURadioMaterial

scene = load_scene(SCENE_WITH_RIS)
scene.frequency = FREQ_HZ

scene.objects["RIS_panel"].radio_material = ITURadioMaterial(
    name="metal_ris",
    itu_type="metal",
    thickness=0.01
)

In [ ]:
import os
import numpy as np
import mitsuba as mi

try:
    mi.set_variant("cuda_ad_mono_polarized")
except Exception:
    mi.set_variant("llvm_ad_mono_polarized")

from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    PathSolver,
)

# ============================================================
# 1) Scene paths
# ============================================================
SCENE_NO_RIS   = str(_sionna_repo_root() / "scenes" / "University.xml")
SCENE_WITH_RIS = str(_sionna_repo_root() / "scenes" / "University_RIS.xml")

if not os.path.exists(SCENE_NO_RIS):
    raise FileNotFoundError(f"Scene not found: {SCENE_NO_RIS}")

if not os.path.exists(SCENE_WITH_RIS):
    raise FileNotFoundError(f"Scene not found: {SCENE_WITH_RIS}")

# ============================================================
# 2) Simulation parameters
# ============================================================
FREQ_HZ = 1e9
MAX_DEPTH = 4

# User-defined positions
BS_POS = np.array([0.0,   0.0, 10.0], dtype=np.float32)
UE_POS = np.array([-70.0, 180.0, 1.5], dtype=np.float32)

# BS: 8x8 planar array, UE: single antenna
BS_ROWS, BS_COLS = 8, 8
UE_ROWS, UE_COLS = 1, 1

# ============================================================
# 3) Utilities
# ============================================================
def db10(x):
    x = np.maximum(np.real(x), 1e-30)
    return 10*np.log10(x)

def db20(x):
    x = np.maximum(np.abs(x), 1e-30)
    return 20*np.log10(x)

def setup_scene(scene_path, bs_pos, ue_pos):
    """
    Load scene, set carrier frequency, configure arrays,
    and place TX/RX.
    """
    scene = load_scene(scene_path)
    scene.frequency = FREQ_HZ

    # BS transmit array
    scene.tx_array = PlanarArray(
        num_rows=BS_ROWS,
        num_cols=BS_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    # UE receive array
    scene.rx_array = PlanarArray(
        num_rows=UE_ROWS,
        num_cols=UE_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    # Add BS and UE
    scene.add(Transmitter(name="bs", position=bs_pos))
    scene.add(Receiver(name="ue", position=ue_pos))

    return scene

def compute_paths(scene):
    """
    Compute propagation paths.
    """
    solver = PathSolver()
    paths = solver(
        scene,
        max_depth=MAX_DEPTH,
        los=True,
        specular_reflection=True,
        diffraction=True,
        edge_diffraction=False,
        refraction=False,
        diffuse_reflection=False
    )
    return paths

def get_narrowband_channel(paths):
    """
    Extract narrowband channel matrix H from CFR.
    Expected CFR shape:
    [num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, num_subcarriers]
    """
    freqs = np.array([FREQ_HZ], dtype=np.float32)
    h = np.array(paths.cfr(frequencies=freqs, out_type="numpy"))

    print("CFR shape:", h.shape)

    if h.ndim != 6:
        raise RuntimeError(f"Unexpected CFR shape: {h.shape}")

    # Single RX, single TX, one time step, one tone
    H = h[0, :, 0, :, 0, 0]

    print("Extracted H shape:", H.shape)
    return H

def svd_beamforming(H):
    """
    Dominant singular vectors for TX/RX beamforming.
    """
    U, S, Vh = np.linalg.svd(H)
    w_rx = U[:, 0]
    w_tx = Vh.conj().T[:, 0]
    sigma_max = S[0]
    return w_tx, w_rx, sigma_max

def metrics_from_channel(H, w_tx, w_rx):
    """
    Compare no-BF and BF effective power.
    """
    power_no_bf = np.linalg.norm(H, "fro")**2

    # Effective scalar channel after beamforming
    h_eff = np.vdot(w_rx, H @ w_tx)
    power_bf = np.abs(h_eff)**2

    return power_no_bf, power_bf, h_eff

def count_paths(paths):
    """
    Try to count how many paths are available.
    Depending on Sionna version, path tensors may differ.
    """
    try:
        a, tau = paths.cir(out_type="numpy")
        # Typical CIR shape:
        # a: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
        if a.ndim >= 5:
            return a.shape[4]
        return None
    except Exception:
        return None

def evaluate_scene(scene_path, label):
    """
    Full pipeline for one scene.
    """
    print(f"\n==============================")
    print(f"Evaluating: {label}")
    print(f"Scene file : {scene_path}")
    print(f"==============================")

    scene = setup_scene(scene_path, BS_POS, UE_POS)
    paths = compute_paths(scene)
    H = get_narrowband_channel(paths)

    w_tx, w_rx, sigma_max = svd_beamforming(H)
    p_no_bf, p_bf, h_eff = metrics_from_channel(H, w_tx, w_rx)
    n_paths = count_paths(paths)

    return {
        "label": label,
        "scene": scene,
        "paths": paths,
        "H": H,
        "w_tx": w_tx,
        "w_rx": w_rx,
        "sigma_max": sigma_max,
        "power_no_bf": p_no_bf,
        "power_bf": p_bf,
        "h_eff": h_eff,
        "num_paths": n_paths,
    }

# ============================================================
# 4) Run both cases
# ============================================================
res_no_ris = evaluate_scene(SCENE_NO_RIS, "No RIS")
res_ris    = evaluate_scene(SCENE_WITH_RIS, "With RIS")

# ============================================================
# 5) Print comparison
# ============================================================
for res in [res_no_ris, res_ris]:
    print(f"\n=== {res['label']} ===")
    print("H shape               :", res["H"].shape)
    print("num_paths             :", res["num_paths"])
    print("sigma_max             :", res["sigma_max"])
    print("||H||_F^2 [dB]        :", db10(res["power_no_bf"]))
    print("|h_eff|^2 [dB]        :", db10(res["power_bf"]))
    print("|h_eff| [dB]          :", db20(res["h_eff"]))
    print("Beamforming gain [dB] :", db10(res["power_bf"]) - db10(res["power_no_bf"]))

# RIS improvement after BF
ris_gain_bf_db = db10(res_ris["power_bf"]) - db10(res_no_ris["power_bf"])
ris_gain_nb_db = db10(res_ris["power_no_bf"]) - db10(res_no_ris["power_no_bf"])

print("\n======================================")
print("Comparison: With RIS vs No RIS")
print("======================================")
print("No-BF power improvement [dB] :", ris_gain_nb_db)
print("BF power improvement    [dB] :", ris_gain_bf_db)

# ============================================================
# 6) Optional preview
# ============================================================
# Uncomment if you want to visually inspect the scenes
# res_no_ris["scene"].preview()
# res_ris["scene"].preview()

In [ ]:
scene_test = load_scene(SCENE_WITH_RIS)
scene_test.frequency = FREQ_HZ

print("Objects:", scene_test.objects.keys())
print("Materials:", scene_test.radio_materials.keys())

if "RIS_panel" in scene_test.objects:
    print("RIS_panel material:", scene_test.objects["RIS_panel"].radio_material)
else:
    print("RIS_panel not found in scene.objects")

In [ ]:
import os
import numpy as np
import mitsuba as mi

try:
    mi.set_variant("cuda_ad_mono_polarized")
except Exception:
    mi.set_variant("llvm_ad_mono_polarized")

from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    PathSolver,
)

SCENE_NO_RIS   = str(_sionna_repo_root() / "scenes" / "University.xml")
SCENE_WITH_RIS = str(_sionna_repo_root() / "scenes" / "University_RIS.xml")

FREQ_HZ = 28e9
MAX_DEPTH = 5

BS_POS = np.array([0.0, 0.0, 10.0], dtype=np.float32)
UE_POS = np.array([-70.0, 180.0, 1.5], dtype=np.float32)

BS_ROWS, BS_COLS = 8, 8
UE_ROWS, UE_COLS = 1, 1

def db10(x):
    x = np.maximum(np.real(x), 1e-30)
    return 10*np.log10(x)

def setup_scene(scene_path, bs_pos, ue_pos, tune_ris=False):
    scene = load_scene(scene_path)
    scene.frequency = FREQ_HZ

    scene.tx_array = PlanarArray(
        num_rows=BS_ROWS,
        num_cols=BS_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    scene.rx_array = PlanarArray(
        num_rows=UE_ROWS,
        num_cols=UE_COLS,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",
        polarization="V"
    )

    scene.add(Transmitter(name="bs", position=bs_pos))
    scene.add(Receiver(name="ue", position=ue_pos))

    # RIS를 Python에서 더 공격적으로 조정
    if tune_ris and "RIS_panel" in scene.objects:
        ris = scene.get("RIS_panel")

        # 위치는 네가 정한 좌표 유지
        ris.position = mi.Point3f([-113.0, 115.0, 5.0])

        # 매우 크게 키워서 먼저 '효과가 보이는지' 확인
        ris.scaling = mi.Vector3f([8.0, 8.0, 8.0])

        # 일단 UE 쪽을 보게 함
        # SceneObject.look_at()은 object's x-axis가 target을 향하도록 설정함
        ris.look_at(mi.Point3f(UE_POS.tolist()))

        print("RIS tuned in Python:")
        print("  position :", ris.position)
        print("  scaling  :", ris.scaling)
        print("  orient   :", ris.orientation)
        print("  obj_id   :", ris.object_id)

    return scene

def compute_paths(scene, use_diffraction=False):
    solver = PathSolver()
    paths = solver(
        scene,
        max_depth=MAX_DEPTH,
        los=True,
        specular_reflection=True,
        diffraction=use_diffraction,
        edge_diffraction=False,
        refraction=False,
        diffuse_reflection=False
    )
    return paths

def get_narrowband_channel(paths):
    freqs = np.array([FREQ_HZ], dtype=np.float32)
    h = np.array(paths.cfr(frequencies=freqs, out_type="numpy"))
    if h.ndim != 6:
        raise RuntimeError(f"Unexpected CFR shape: {h.shape}")
    return h[0, :, 0, :, 0, 0]

def svd_beamforming(H):
    U, S, Vh = np.linalg.svd(H)
    return Vh.conj().T[:, 0], U[:, 0], S[0]

def metrics_from_channel(H, w_tx_opt, w_rx_opt):
    Nr, Nt = H.shape

    # no-BF: equal-gain transmission
    w_tx_eq = np.ones(Nt, dtype=np.complex64) / np.sqrt(Nt)
    w_rx_eq = np.ones(Nr, dtype=np.complex64) / np.sqrt(Nr)

    h_no_bf = np.vdot(w_rx_eq, H @ w_tx_eq)
    p_no_bf = np.abs(h_no_bf)**2

    h_bf = np.vdot(w_rx_opt, H @ w_tx_opt)
    p_bf = np.abs(h_bf)**2

    return p_no_bf, p_bf

def inspect_paths(scene, paths, label):
    print(f"\n--- Path inspection: {label} ---")

    # valid path mask
    valid = np.array(paths.valid, dtype=bool)
    print("valid shape:", valid.shape)
    print("num valid  :", valid.sum())

    objects = np.array(paths.objects)
    interactions = np.array(paths.interactions)
    vertices = np.array(paths.vertices)

    print("objects shape      :", objects.shape)
    print("interactions shape :", interactions.shape)
    print("vertices shape     :", vertices.shape)

    ris_id = None
    if "RIS_panel" in scene.objects:
        ris_id = scene.get("RIS_panel").object_id
        print("RIS object_id:", ris_id)

    # 현재 구조에서는 [max_depth, rx, rx_ant, tx, tx_ant, num_paths] 형태일 가능성이 큼
    # 여기서는 단일 rx/tx 기준으로 마지막 path 축만 훑음
    max_depth = objects.shape[0]
    num_paths = objects.shape[-1]

    for p in range(num_paths):
        used_objs = []
        used_ints = []
        used_verts = []

        for d in range(max_depth):
            oid = objects[d, 0, 0, 0, 0, p]
            itp = interactions[d, 0, 0, 0, 0, p]
            vtx = vertices[d, 0, 0, 0, 0, p]

            used_objs.append(int(oid))
            used_ints.append(int(itp))
            used_verts.append(vtx)

        print(f"\nPath {p}:")
        print("  object ids    :", used_objs)
        print("  interactions  :", used_ints)
        print("  vertices      :", used_verts)

        if ris_id is not None and ris_id in used_objs:
            print("  --> This path uses RIS_panel")

def evaluate(scene_path, label, tune_ris=False, use_diffraction=False):
    print("\n==============================")
    print(f"Evaluating: {label}")
    print("==============================")

    scene = setup_scene(scene_path, BS_POS, UE_POS, tune_ris=tune_ris)
    print("Objects:", scene.objects.keys())

    paths = compute_paths(scene, use_diffraction=use_diffraction)
    inspect_paths(scene, paths, label)

    H = get_narrowband_channel(paths)
    w_tx, w_rx, sigma_max = svd_beamforming(H)
    p_no_bf, p_bf = metrics_from_channel(H, w_tx, w_rx)

    return {
        "scene": scene,
        "paths": paths,
        "H": H,
        "sigma_max": sigma_max,
        "power_no_bf": p_no_bf,
        "power_bf": p_bf,
    }

# 1) baseline
res_no = evaluate(SCENE_NO_RIS, "No RIS", tune_ris=False, use_diffraction=False)

# 2) with RIS, XML 그대로
res_ris = evaluate(SCENE_WITH_RIS, "With RIS (as exported)", tune_ris=False, use_diffraction=False)
Q
# 3) with RIS, Python에서 크게/방향 조정
res_ris_tuned = evaluate(SCENE_WITH_RIS, "With RIS (tuned in Python)", tune_ris=True, use_diffraction=False)

print("\n========== Summary ==========")
for name, res in [
    ("No RIS", res_no),
    ("With RIS", res_ris),
    ("With RIS tuned", res_ris_tuned),
]:
    print(f"\n{name}")
    print("sigma_max        :", res["sigma_max"])
    print("No-BF power [dB] :", db10(res["power_no_bf"]))
    print("BF power [dB]    :", db10(res["power_bf"]))

print("\n||H_with - H_no||_F       =", np.linalg.norm(res_ris["H"] - res_no["H"]))
print("||H_with_tuned - H_no||_F =", np.linalg.norm(res_ris_tuned["H"] - res_no["H"]))